In [3]:
from datetime import datetime

def price_commodity_storage(
    injection_dates: list,
    withdrawal_dates: list,
    injection_prices: list,
    withdrawal_prices: list,
    injection_rate: float,
    withdrawal_rate: float,
    max_storage_volume: float,
    monthly_storage_cost: float,
    planned_injection_volumes: list,
    planned_withdrawal_volumes: list
) -> dict:
    """
    Prices a flexible commodity storage contract with multiple injection/withdrawal windows.
    Enforces maximum capacity constraints and physical flow limits.
    """
    # 1. Consolidate and sort all operations chronologically
    ledger = []
    
    for dt, price, vol in zip(injection_dates, injection_prices, planned_injection_volumes):
        ledger.append({'date': datetime.strptime(dt, "%Y-%m-%d"), 'type': 'injection', 'price': price, 'target_vol': vol})
        
    for dt, price, vol in zip(withdrawal_dates, withdrawal_prices, planned_withdrawal_volumes):
        ledger.append({'date': datetime.strptime(dt, "%Y-%m-%d"), 'type': 'withdrawal', 'price': price, 'target_vol': vol})
        
    ledger.sort(key=lambda x: x['date'])
    
    # 2. Track contract duration for storage cost calculation
    if not ledger:
        return {"Contract Value ($)": 0.0}
    
    total_days = (ledger[-1]['date'] - ledger[0]['date']).days
    total_storage_cost = (total_days / 30.44) * monthly_storage_cost

    # 3. Simulate physical operations and cash flows
    current_inventory = 0.0
    total_spent = 0.0
    total_earned = 0.0
    
    for transaction in ledger:
        price = transaction['price']
        target = transaction['target_vol']
        
        if transaction['type'] == 'injection':
            # Flow is limited by planned volume, facility injection speed, and remaining storage room
            actual_injection = min(target, injection_rate, max_storage_volume - current_inventory)
            current_inventory += actual_injection
            total_spent += actual_injection * price
            
        elif transaction['type'] == 'withdrawal':
            # Flow is limited by planned volume, facility withdrawal speed, and gas physically in storage
            actual_withdrawal = min(target, withdrawal_rate, current_inventory)
            current_inventory -= actual_withdrawal
            total_earned += actual_withdrawal * price

    # 4. Final Financial Aggregation
    gross_arbitrage = total_earned - total_spent
    net_contract_value = gross_arbitrage - total_storage_cost
    
    return {
        "Gross Revenue ($)": total_earned,
        "Total Purchase Cost ($)": total_spent,
        "Total Storage Cost ($)": total_storage_cost,
        "Ending Inventory (MMBtu)": current_inventory,
        "Net Contract Value ($)": net_contract_value
    }


In [4]:
# --- Test Implementation ---
if __name__ == "__main__":
    result = price_commodity_storage(
        injection_dates=['2026-04-01', '2026-05-01'],
        withdrawal_dates=['2026-11-01', '2026-12-01'],
        injection_prices=[2.00, 2.10],
        withdrawal_prices=[3.50, 3.80],
        injection_rate=600000,
        withdrawal_rate=600000,
        max_storage_volume=1000000,
        monthly_storage_cost=100000,
        planned_injection_volumes=[500000, 500000],
        planned_withdrawal_volumes=[500000, 500000]
    )
    
    for key, value in result.items():
        print(f"{key}: {value:,.2f}")


Gross Revenue ($): 3,650,000.00
Total Purchase Cost ($): 2,050,000.00
Total Storage Cost ($): 801,576.87
Ending Inventory (MMBtu): 0.00
Net Contract Value ($): 798,423.13
